# 🚗 Car Price Prediction Using Machine Learning
---
**Objective:** Build a regression model to predict car prices using features like brand, mileage, engine power, and fuel type.  
**Dataset:** 19,237 records · 17 features  
**Tools:** Pandas · Scikit-learn · Matplotlib · Seaborn  

 **Workflow:** Data Loading → EDA → Preprocessing → Feature Engineering → Modelling → Evaluation → Insights


In [13]:
import pandas as pd
import numpy as np

In [9]:
df = pd.read_csv('car_price_prediction.csv')

print(f"Shape     : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory    : {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
print()
df.sample(5)

Shape     : 19,237 rows × 18 columns
Memory    : 14.48 MB



,ID,Price,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags
16188,45651677,3450,831,HYUNDAI,Sonata,2011,Sedan,Yes,Hybrid,2.4,224349 km,4.0,Automatic,Front,04-May,Left wheel,Grey,0
6732,36559154,19726,-,MERCEDES-BENZ,Sprinter 411,2002,Goods wagon,No,Diesel,2.2 Turbo,280000 km,4.0,Manual,Rear,02-Mar,Left wheel,White,2
7533,45769134,11290,2377,TOYOTA,Tundra,2016,Jeep,Yes,Petrol,5.7,54875 km,8.0,Automatic,Rear,04-May,Left wheel,Grey,12
11335,45803150,11290,891,HYUNDAI,Sonata,2016,Sedan,Yes,LPG,2,257646 km,4.0,Automatic,Front,04-May,Left wheel,Silver,4
1450,45732339,20789,836,HYUNDAI,Santa FE,2010,Jeep,Yes,Diesel,2,208130 km,4.0,Automatic,Front,04-May,Left wheel,Silver,4


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19237 entries, 0 to 19236
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                19237 non-null  int64  
 1   Price             19237 non-null  int64  
 2   Levy              19237 non-null  object 
 3   Manufacturer      19237 non-null  object 
 4   Model             19237 non-null  object 
 5   Prod. year        19237 non-null  int64  
 6   Category          19237 non-null  object 
 7   Leather interior  19237 non-null  object 
 8   Fuel type         19237 non-null  object 
 9   Engine volume     19237 non-null  object 
 10  Mileage           19237 non-null  object 
 11  Cylinders         19237 non-null  float64
 12  Gear box type     19237 non-null  object 
 13  Drive wheels      19237 non-null  object 
 14  Doors             19237 non-null  object 
 15  Wheel             19237 non-null  object 
 16  Color             19237 non-null  object

In [11]:
df.describe().round(2)

,ID,Price,Prod. year,Cylinders,Airbags
count,19237.00,19237.00,19237.00,19237.00,19237.00
mean,45576535.89,18555.93,2010.91,4.58,6.58
std,936591.42,190581.27,5.67,1.20,4.32
min,20746880.00,1.00,1939.00,1.00,0.00
25%,45698374.00,5331.00,2009.00,4.00,4.00
50%,45772308.00,13172.00,2012.00,4.00,6.00
75%,45802036.00,22075.00,2015.00,4.00,12.00
max,45816654.00,26307500.00,2020.00,16.00,16.00


In [21]:
# Check for missing values and special placeholders
print("=== Null Counts ===")
print(df.isnull().sum())
print()
print("=== Levy unique sample (first 10) ===")
print(df['Levy'].unique()[:10])
print()
print("=== Mileage sample ===")
print(df['Mileage'].head(5).tolist())
print()
print("=== Doors unique values ===")
print(df['Doors'].unique())

=== Null Counts ===
Price            0
Levy             0
Manufacturer     0
Model            0
Prod. year       0
Category         0
Fuel type        0
Engine volume    0
Mileage          0
Cylinders        0
Gear box type    0
Drive wheels     0
Doors            0
Wheel            0
Color            0
Airbags          0
Leather_Bin      0
dtype: int64

=== Levy unique sample (first 10) ===
[1399. 1018.    0.  862.  446.  891.  761.  751.  394. 1053.]

=== Mileage sample ===
[186005.0, 192000.0, 200000.0, 168966.0, 91901.0]

=== Doors unique values ===
[4. 2.]


In [16]:
df = pd.read_csv('car_price_prediction.csv')
df = df.drop(columns=['ID'])

# ── Price ──────────────────────────────────────────
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
df = df[df['Price'] < df['Price'].quantile(0.99)]   # remove top 1% outliers
df = df[df['Price'] > 100]                           # remove junk prices

# ── Levy (tax): replace '-' with 0 ─────────────────
df['Levy'] = df['Levy'].replace('-', '0').astype(float)

# ── Mileage: strip ' km' and convert ───────────────
df['Mileage'] = (df['Mileage']
                 .str.replace(' km', '', regex=False)
                 .str.replace(',', '', regex=False)
                 .astype(float))

# ── Engine volume: coerce to numeric ───────────────
df['Engine volume'] = pd.to_numeric(df['Engine volume'], errors='coerce')

# ── Doors: '04-May' style → integer ────────────────
def fix_doors(val):
    try:
        return int(str(val).split('-')[0])
    except:
        return np.nan

df['Doors'] = df['Doors'].apply(fix_doors)

# ── Leather interior: binary ────────────────────────
df['Leather_Bin'] = (df['Leather interior'] == 'Yes').astype(int)
df = df.drop(columns=['Leather interior'])

# ── Drop remaining NaNs ─────────────────────────────
df = df.dropna()

print(f"✅ Clean dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print()
print("Sample after cleaning:")
df.head(3)

✅ Clean dataset: 16,754 rows × 17 columns

Sample after cleaning:


,Price,Levy,Manufacturer,Model,Prod. year,Category,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags,Leather_Bin
0,13328,1399.0,LEXUS,RX 450,2010,Jeep,Hybrid,3.5,186005.0,6.0,Automatic,4x4,4.0,Left wheel,Silver,12,1
1,16621,1018.0,CHEVROLET,Equinox,2011,Jeep,Petrol,3.0,192000.0,6.0,Tiptronic,4x4,4.0,Left wheel,Black,8,0
2,8467,0.0,HONDA,FIT,2006,Hatchback,Petrol,1.3,200000.0,4.0,Variator,Front,4.0,Right-hand drive,Black,2,0


In [20]:
df.to_csv("Cleaned_dataset.csv")
print(f"✅ Data Exported Successfully")

✅ Data Exported Successfully
